# Reproduce NegLabel + TANL on ImageNet
Run cells top to bottom. Should take ~30 mins total on T4 GPU.

**Expected results:**
- NegLabel FPR95 ≈ 25.40%
- TANL FPR95 ≈ 9.81%

## Cell 1 — Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Go to Runtime > Change runtime type > T4'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ood-vlm-research'
import os
os.makedirs(DRIVE, exist_ok=True)
print('Drive mounted at', DRIVE)

## Cell 3 — Clone OpenOOD-VLM

In [ ]:
import os
REPO = '/content/OpenOOD-VLM'
if not os.path.exists(REPO):
    !git clone https://github.com/YBZh/OpenOOD-VLM {REPO}
else:
    !cd {REPO} && git pull
%cd {REPO}
print('Repo ready at', REPO)

## Cell 4 — Install Dependencies

In [ ]:
!pip install -q -e . --no-deps
!pip install -q faiss-cpu ipdb
import openood
print('OpenOOD-VLM installed ✅')

## Cell 5 — Download Datasets
Downloads to Drive so you never repeat this step.
Skip if already downloaded (checks Drive first).

In [ ]:
import os, gdown, zipfile, shutil

DRIVE_DATA = f'{DRIVE}/data'
LOCAL_DATA = f'{REPO}/data'
os.makedirs(DRIVE_DATA, exist_ok=True)
os.makedirs(LOCAL_DATA, exist_ok=True)

# ----- Helper -----
def download_and_extract(name, gid, dest_dir, subfolder=None):
    final_path = os.path.join(dest_dir, subfolder or name)
    if os.path.exists(final_path):
        print(f'{name} already exists ✅')
        return
    print(f'Downloading {name}...')
    zip_path = os.path.join(dest_dir, name + '.zip')
    gdown.download(id=gid, output=zip_path, fuzzy=True, quiet=False)
    print(f'Extracting {name}...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(dest_dir)
    os.remove(zip_path)
    print(f'{name} done ✅')

# ----- benchmark_imglist -----
imglist_dir = os.path.join(DRIVE_DATA, 'benchmark_imglist')
if not os.path.exists(imglist_dir):
    print('Downloading benchmark_imglist...')
    zip_path = os.path.join(DRIVE_DATA, 'benchmark_imglist.zip')
    gdown.download(id='1XKzBdWCqg3vPoj-D32YixJyJJ0hL63gP', output=zip_path, fuzzy=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DRIVE_DATA)
    os.remove(zip_path)
    print('benchmark_imglist done ✅')
else:
    print('benchmark_imglist already exists ✅')

# ----- Image Datasets -----
img_dir = os.path.join(DRIVE_DATA, 'images_largescale')
os.makedirs(img_dir, exist_ok=True)

datasets = [
    ('imagenet_1k',  '1i1ipLDFARR-JZ9argXd2-0a6DXwVhXEj', 'val'),
    ('inaturalist',  '1zfLfMvoUD0CUlKNnkk7LgxZZBnTBipdj', 'inaturalist'),
    ('sun',          '1ISK0STxWzWmg-_uUr4RQ8GSLFW7TZiKp', 'sun'),
    ('places',       '1fZ8TbPC4JGqUCm-VtvrmkYxqRNp2PoB3', 'places'),
    ('texture',      '1OSz1m3hHfVWbRdmMwKbUzoU8Hg9UKcam', 'dtd'),
]

for name, gid, check_folder in datasets:
    check_path = os.path.join(img_dir, check_folder)
    if os.path.exists(check_path):
        print(f'{name} already exists ✅')
        continue
    print(f'Downloading {name}...')
    zip_path = os.path.join(img_dir, name + '.zip')
    gdown.download(id=gid, output=zip_path, fuzzy=True, quiet=False)
    print(f'Extracting {name}...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(img_dir)
    os.remove(zip_path)
    print(f'{name} done ✅')

print('\nAll datasets ready on Drive ✅')

## Cell 6 — Link Drive Data to Repo
Symlink Drive data into the repo's data/ folder so scripts find it.

In [ ]:
import os

DRIVE_DATA = f'{DRIVE}/data'
LOCAL_DATA = f'{REPO}/data'

# Symlink benchmark_imglist
src = os.path.join(DRIVE_DATA, 'benchmark_imglist')
dst = os.path.join(LOCAL_DATA, 'benchmark_imglist')
if not os.path.exists(dst):
    os.symlink(src, dst)
    print('benchmark_imglist linked ✅')
else:
    print('benchmark_imglist already linked ✅')

# Symlink images_largescale
src = os.path.join(DRIVE_DATA, 'images_largescale')
dst = os.path.join(LOCAL_DATA, 'images_largescale')
if not os.path.exists(dst):
    os.symlink(src, dst)
    print('images_largescale linked ✅')
else:
    print('images_largescale already linked ✅')

# Fix expected folder structure inside images_largescale
img_dir = os.path.join(DRIVE_DATA, 'images_largescale')

# inaturalist expects inaturalist/images/ subfolder
inat_images = os.path.join(img_dir, 'inaturalist', 'images')
if not os.path.exists(inat_images):
    os.makedirs(os.path.join(img_dir, 'inaturalist'), exist_ok=True)
    # inaturalist images are directly in inaturalist/ folder after extraction
    # create images/ subfolder symlink or move
    inat_root = os.path.join(img_dir, 'inaturalist')
    # check if images are directly in inaturalist/
    files = [f for f in os.listdir(inat_root) if f.endswith('.jpg')]
    if files:
        # images are directly here, create images/ subfolder
        os.makedirs(inat_images, exist_ok=True)
        for f in os.listdir(inat_root):
            if f.endswith('.jpg'):
                os.rename(os.path.join(inat_root, f), os.path.join(inat_images, f))
        print('inaturalist/images/ created ✅')
    else:
        print('inaturalist structure unclear — check manually')
else:
    print('inaturalist/images/ already exists ✅')

# texture expects texture/ pointing to dtd/images/
texture_link = os.path.join(img_dir, 'texture')
if not os.path.exists(texture_link):
    os.symlink(os.path.join(img_dir, 'dtd', 'images'), texture_link)
    print('texture symlink created ✅')
else:
    print('texture already exists ✅')

# Verify all expected paths exist
checks = [
    os.path.join(img_dir, 'val'),           # ImageNet
    os.path.join(img_dir, 'inaturalist', 'images'),
    os.path.join(img_dir, 'sun'),
    os.path.join(img_dir, 'places'),
    os.path.join(img_dir, 'texture'),
]
for p in checks:
    status = '✅' if os.path.exists(p) else '❌ MISSING'
    print(f'{status}  {p}')

## Cell 7 — Download Missing imglist Files
The benchmark_imglist from Drive may be missing test_sun.txt, test_places.txt, test_dtd.txt.
We generate them from what we have.

In [ ]:
import os, glob

imglist_dir = os.path.join(DRIVE_DATA, 'benchmark_imglist', 'imagenet')
img_dir = os.path.join(DRIVE_DATA, 'images_largescale')

def make_imglist(folder, out_txt, label=-1, extensions=('.jpg', '.jpeg', '.png', '.JPEG')):
    """Generate imglist from all images in a folder."""
    if os.path.exists(out_txt):
        print(f'{os.path.basename(out_txt)} already exists ✅')
        return
    lines = []
    for root, dirs, files in os.walk(folder):
        dirs.sort()
        for f in sorted(files):
            if f.lower().endswith(extensions):
                rel = os.path.relpath(os.path.join(root, f), img_dir)
                lines.append(f'{rel} {label}')
    with open(out_txt, 'w') as fp:
        fp.write('\n'.join(lines))
    print(f'{os.path.basename(out_txt)} generated with {len(lines)} images ✅')

# Generate missing files
make_imglist(
    folder=os.path.join(img_dir, 'sun'),
    out_txt=os.path.join(imglist_dir, 'test_sun.txt')
)
make_imglist(
    folder=os.path.join(img_dir, 'places'),
    out_txt=os.path.join(imglist_dir, 'test_places.txt')
)
make_imglist(
    folder=os.path.join(img_dir, 'dtd', 'images'),
    out_txt=os.path.join(imglist_dir, 'test_dtd.txt')
)

# Verify all required files exist
required = ['test_imagenet.txt', 'test_inaturalist.txt', 'test_sun.txt',
            'test_places.txt', 'test_dtd.txt']
for f in required:
    p = os.path.join(imglist_dir, f)
    status = '✅' if os.path.exists(p) else '❌ MISSING'
    if os.path.exists(p):
        count = sum(1 for _ in open(p))
        print(f'{status}  {f}  ({count} images)')
    else:
        print(f'{status}  {f}')

## Cell 8 — Download Corpus Dataset (WordNet labels for NegLabel/TANL)

In [ ]:
import os, gdown

# Check if corpus already exists in the repo
corpus_paths = [
    f'{REPO}/data/corpus_datasets',
    f'{REPO}/corpus_datasets',
]

corpus_exists = any(os.path.exists(p) for p in corpus_paths)
if corpus_exists:
    print('Corpus dataset already exists ✅')
else:
    # Check if it's bundled in the repo
    !find {REPO} -name '*.txt' | grep -i corpus | head -5
    !find {REPO} -name 'wordnet*' | head -5
    print('Checking repo for corpus files...')

## Cell 9 — Verify Full Setup

In [ ]:
import os

print('=== Setup Verification ===')
print()

checks = {
    'OpenOOD-VLM repo': REPO,
    'ImageNet val':      f'{DRIVE}/data/images_largescale/val',
    'iNaturalist':       f'{DRIVE}/data/images_largescale/inaturalist/images',
    'SUN':               f'{DRIVE}/data/images_largescale/sun',
    'Places':            f'{DRIVE}/data/images_largescale/places',
    'Textures (dtd)':    f'{DRIVE}/data/images_largescale/dtd',
    'benchmark_imglist': f'{DRIVE}/data/benchmark_imglist/imagenet',
    'test_imagenet.txt': f'{DRIVE}/data/benchmark_imglist/imagenet/test_imagenet.txt',
    'test_sun.txt':      f'{DRIVE}/data/benchmark_imglist/imagenet/test_sun.txt',
    'test_places.txt':   f'{DRIVE}/data/benchmark_imglist/imagenet/test_places.txt',
    'test_dtd.txt':      f'{DRIVE}/data/benchmark_imglist/imagenet/test_dtd.txt',
}

all_ok = True
for name, path in checks.items():
    ok = os.path.exists(path)
    if not ok:
        all_ok = False
    print(f'{"✅" if ok else "❌"}  {name}')

print()
if all_ok:
    print('All checks passed! Ready to run experiments ✅')
else:
    print('Some checks failed ❌ — fix before running experiments')

## Cell 10 — Run NegLabel (Baseline)
Expected: FPR95 ≈ 25.40% on ImageNet benchmark

In [ ]:
import os
%cd {REPO}

!python main.py \
    --config configs/datasets/imagenet/imagenet_traditional_four_ood.yml \
    configs/networks/fixed_clip.yml \
    configs/pipelines/test/test_fsood.yml \
    configs/preprocessors/base_preprocessor.yml \
    configs/postprocessors/mcm.yml \
    --dataset.train.batch_size 128 \
    --dataset.train.few_shot 0 \
    --dataset.num_classes 11000 \
    --evaluator.name ood_clip \
    --network.name fixedclip_negoodprompt \
    --network.backbone.ood_number 10000 \
    --network.backbone.text_prompt nice \
    --network.backbone.text_center True \
    --network.pretrained False \
    --postprocessor.APS_mode False \
    --postprocessor.name oneoodpromptdevelop \
    --postprocessor.postprocessor_args.group_num 100 \
    --postprocessor.postprocessor_args.random_permute True \
    --postprocessor.postprocessor_args.tau 1.0 \
    --postprocessor.postprocessor_args.beta 1.0 \
    --postprocessor.postprocessor_args.in_score sum \
    --num_gpus 1 --num_workers 4 \
    --merge_option merge \
    --output_dir ./results/neglabel/ \
    --mark neglabel_reproduction

## Cell 11 — Run TANL
Expected: FPR95 ≈ 9.81% on ImageNet benchmark

In [ ]:
%cd {REPO}
!sh scripts/ood/TANL/official.sh

## Cell 12 — Save Results to Drive

In [ ]:
import shutil, os

results_drive = f'{DRIVE}/results'
os.makedirs(results_drive, exist_ok=True)

# Copy results to Drive
shutil.copytree(
    f'{REPO}/results',
    results_drive,
    dirs_exist_ok=True
)
print(f'Results saved to {results_drive} ✅')
print(os.listdir(results_drive))